# Convex Sets, Convex Functions, and Global Optimality
### MECH 559 - Systems Optimization - L05

This notebook demonstrates the convexity material from the lecture: testing sets and functions for convexity directly from their definitions, the closure properties (intersection vs. union, sums of convex functions), and the payoff - why a stationary point of a convex function is automatically a *global* minimizer.

## Learning objectives

By the end, students should be able to:

1. Test whether a set is convex using the segment definition, and recognize why "concave set" is not a meaningful term.
2. Test whether a function is convex using the chord inequality, and connect it to the Hessian positive-semidefiniteness test.
3. Demonstrate that intersections of convex sets are convex while unions generally are not, and that sums of convex functions are convex.
4. Show numerically that a convex function's stationary point satisfies the global lower-bound inequality $f(x)\ge f(x_o)+\nabla f(x_o)^{\mathrm T}(x-x_o)$ for every $x$.

> **Classroom rhythm:** pause at each **Predict** prompt, collect answers, then run the next cell.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

try:
    import ipywidgets as widgets
    from IPython.display import display
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams.update({
    "font.size": 11,
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

COLORS = {"boundary": "#00798C", "optimum": "#D1495B", "accent": "#EDAE49"}

print(f"NumPy {np.__version__}; optional widgets available: {WIDGETS_AVAILABLE}")

---
## 1. Convex sets: the segment test

$\mathcal S\subseteq\mathbb R^n$ is convex iff for every $x,y\in\mathcal S$ and every $\alpha\in[0,1]$, the point $z=\alpha x+(1-\alpha)y$ is also in $\mathcal S$. We test this by sampling many point pairs and checking the whole segment between them.

> **Predict:** two overlapping disks - is their *intersection* convex? Is their *union*?

In [ ]:
def in_disk(p, center, radius):
    return np.linalg.norm(p - center) <= radius

def segment_test(membership_fn, box=(-3, 3), n_trials=400, n_pts=25, seed=0):
    rng = np.random.default_rng(seed)
    for _ in range(n_trials):
        x = rng.uniform(*box, size=2)
        y = rng.uniform(*box, size=2)
        if not (membership_fn(x) and membership_fn(y)):
            continue
        for alpha in np.linspace(0, 1, n_pts):
            if not membership_fn(alpha * x + (1 - alpha) * y):
                return False
    return True

c1, r1 = np.array([-0.4, 0.0]), 1.0
c2, r2 = np.array([0.4, 0.0]), 1.0
intersection = lambda p: in_disk(p, c1, r1) and in_disk(p, c2, r2)
print("Intersection of two overlapping disks convex?", segment_test(intersection))

c1u, r1u = np.array([-1.5, 0.0]), 1.0
c2u, r2u = np.array([1.5, 0.0]), 1.0
union = lambda p: in_disk(p, c1u, r1u) or in_disk(p, c2u, r2u)
x_demo, y_demo = np.array([-2.0, 0.0]), np.array([2.0, 0.0])
mid_demo = 0.5 * x_demo + 0.5 * y_demo
print("Union of two disjoint disks convex? ", union(mid_demo),
      " (midpoint of two feasible points lies in neither disk)")

theta = np.linspace(0, 2*np.pi, 200)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for center, radius in [(c1, r1), (c2, r2)]:
    axes[0].plot(center[0] + radius*np.cos(theta), center[1] + radius*np.sin(theta), color=COLORS["boundary"])
axes[0].set_title("Intersection of overlapping disks: convex")
for center, radius in [(c1u, r1u), (c2u, r2u)]:
    axes[1].plot(center[0] + radius*np.cos(theta), center[1] + radius*np.sin(theta), color=COLORS["boundary"])
axes[1].scatter(*x_demo, color=COLORS["optimum"])
axes[1].scatter(*y_demo, color=COLORS["optimum"])
axes[1].plot([x_demo[0], y_demo[0]], [x_demo[1], y_demo[1]], color=COLORS["accent"], ls="--")
axes[1].scatter(*mid_demo, color="black", marker="x", s=80, label="midpoint: outside both disks")
axes[1].set_title("Union of disjoint disks: not convex")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.set_aspect("equal"); ax.set_xlim(-3, 3); ax.set_ylim(-2, 2)
plt.tight_layout()
plt.show()

---
## 2. Convex functions: the chord test

$f:\mathcal S\to\mathbb R$ on a convex set $\mathcal S$ is convex iff

$$
f(\alpha x+(1-\alpha)y) \le \alpha f(x) + (1-\alpha)f(y) \qquad \forall\, x,y\in\mathcal S,\ \alpha\in[0,1]:
$$

the curve never lies above the straight chord joining two of its points. We test $f(x)=x^2$ (convex everywhere) against $f(x)=x^3$ (convex only for $x\ge0$).

In [ ]:
def chord_test(func, domain, n_trials=1000, seed=0):
    rng = np.random.default_rng(seed)
    violations = 0
    for _ in range(n_trials):
        x, y = rng.uniform(*domain, size=2)
        alpha = rng.uniform(0, 1)
        if func(alpha * x + (1 - alpha) * y) > alpha * func(x) + (1 - alpha) * func(y) + 1e-9:
            violations += 1
    return violations

print("x^2 on [-3,3]: chord violations =", chord_test(lambda x: x**2, (-3, 3)), "(expect 0: globally convex)")
print("x^3 on [-3,3]: chord violations =", chord_test(lambda x: x**3, (-3, 3)), "(expect >0: not globally convex)")
print("x^3 on [0,3]:  chord violations =", chord_test(lambda x: x**3, (0, 3)), "(expect 0: convex for x>=0)")

xx = np.linspace(-3, 3, 300)
x_a, x_b = -2.0, 2.5
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
titles = ["$f(x)=x^2$: convex", "$f(x)=x^3$: not globally convex"]
for ax, func, title in zip(axes, [lambda x: x**2, lambda x: x**3], titles):
    ax.plot(xx, func(xx), color=COLORS["boundary"], lw=2)
    ax.plot([x_a, x_b], [func(x_a), func(x_b)], color=COLORS["accent"], lw=2, ls="--", label="chord")
    ax.set_title(title)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 3. Linking convexity to the Hessian (cross-check with the previous notebook)

A twice-differentiable $f$ is convex on a convex set iff its Hessian is positive semi-definite everywhere on that set. Re-using the three quadratic examples from the Hessians/SOSC notebook: $f_1$ (PD) and $f_2$ (PSD) should pass the chord test everywhere, while $f_3$ (indefinite) should fail it.

In [ ]:
def chord_test_2d(A_mat, b_vec, box=(-3, 3), n_trials=500, seed=0):
    rng = np.random.default_rng(seed)
    def func(x):
        return 0.5 * x @ A_mat @ x + b_vec @ x
    violations = 0
    for _ in range(n_trials):
        x = rng.uniform(*box, size=2)
        y = rng.uniform(*box, size=2)
        alpha = rng.uniform(0, 1)
        if func(alpha*x + (1-alpha)*y) > alpha*func(x) + (1-alpha)*func(y) + 1e-9:
            violations += 1
    return violations

quadratics = {
    "f1 (Hessian positive definite)": (np.array([[2., -3.], [-3., 8.]]), np.array([1., -1.])),
    "f2 (Hessian positive semi-definite)": (np.array([[8., -4.], [-4., 2.]]), np.array([-4., 2.])),
    "f3 (Hessian indefinite)": (np.array([[-2., 0.], [0., 2.]]), np.array([0., 0.])),
}
for name, (A_i, b_i) in quadratics.items():
    print(f"{name}: chord violations = {chord_test_2d(A_i, b_i)}")

---
## 4. Closure property: sum of convex functions is convex

If $f_1$ and $f_2$ are convex, then $f_1+f_2$ is convex. We check this by adding $f_1$ and $f_2$ from above (one PD, one PSD - both convex) and re-running the chord test on the sum.

In [ ]:
A1, b1 = quadratics["f1 (Hessian positive definite)"]
A2, b2 = quadratics["f2 (Hessian positive semi-definite)"]
A_sum, b_sum = A1 + A2, b1 + b2
print("sum's Hessian A1+A2 =\n", A_sum)
print("sum's chord violations =", chord_test_2d(A_sum, b_sum), "(expect 0)")

---
## 5. Why convexity matters: a stationary point becomes a *global* minimizer

For convex $f$, the first-order tangent-plane approximation at any point $x_o$ is a **global underestimator**:

$$
f(x) \ge f(x_o) + \nabla f(x_o)^{\mathrm T}(x-x_o) \qquad \forall x \in \mathcal S.
$$

At a stationary point $x^\ast$ (where $\nabla f(x^\ast)=0$), this collapses to $f(x)\ge f(x^\ast)$ for **every** $x\in\mathcal S$ - not just nearby ones. We verify this for $f_1$: find its (unique) stationary point, then confirm the inequality holds for many randomly sampled points across the domain.

In [ ]:
A1, b1 = quadratics["f1 (Hessian positive definite)"]
def f1_func(x):
    return 0.5 * x @ A1 @ x + b1 @ x

x_star = np.linalg.solve(A1, -b1)
grad_star = A1 @ x_star + b1
print("stationary point x* =", x_star, " grad(x*) =", grad_star, "(should be ~0)")

rng = np.random.default_rng(2)
holds_everywhere = True
for _ in range(500):
    x_sample = rng.uniform(-10, 10, size=2)
    lower_bound = f1_func(x_star) + grad_star @ (x_sample - x_star)
    if f1_func(x_sample) < lower_bound - 1e-9:
        holds_everywhere = False
        break
print("global underestimator f(x) >= f(x*)+grad(x*)'(x-x*) holds for all sampled x:", holds_everywhere)
print(f"f(x*) = {f1_func(x_star):.4f}  (this is a GLOBAL minimum, not just local, because f1 is convex)")

---
## 6. Interactive: chord test for $f(x)=ax^2$

Move $a$ negative and watch the chord inequality break - the visible signature of losing convexity.

In [ ]:
def convexity_lab(a=1.0, x=-2.0, y=2.0, alpha=0.5):
    func = lambda t: a * t**2
    z = alpha * x + (1 - alpha) * y
    lhs = func(z)
    rhs = alpha * func(x) + (1 - alpha) * func(y)
    xx = np.linspace(min(x, y) - 1, max(x, y) + 1, 200)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(xx, func(xx), color=COLORS["boundary"], lw=2, label="$f(x)=ax^2$")
    ax.plot([x, y], [func(x), func(y)], color=COLORS["accent"], ls="--", label="chord")
    ax.scatter([z], [lhs], color=COLORS["optimum"], zorder=3, label="$f(\\alpha x+(1-\\alpha)y)$")
    ax.scatter([z], [rhs], color="black", marker="x", zorder=3, label="$\\alpha f(x)+(1-\\alpha)f(y)$")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    print(f"f(alpha x+(1-alpha) y) = {lhs:.4f}   alpha f(x)+(1-alpha) f(y) = {rhs:.4f}")
    print("chord inequality holds:", lhs <= rhs + 1e-9, "  (try a negative a to break it)")

if WIDGETS_AVAILABLE:
    display(widgets.interactive(convexity_lab,
                                 a=widgets.FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.25),
                                 x=widgets.FloatSlider(value=-2.0, min=-5, max=5, step=0.5),
                                 y=widgets.FloatSlider(value=2.0, min=-5, max=5, step=0.5),
                                 alpha=widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05)))
else:
    print("ipywidgets is not installed; running the default case.")
    convexity_lab()

---
## Takeaways

1. Convexity of a set and convexity of a function are both defined by "segment/chord stays inside/below" - simple to test by sampling, even before you can prove it analytically.
2. Intersections of convex sets are always convex; unions generally are not; sums of convex functions are always convex.
3. The Hessian-PSD test and the chord test are equivalent for twice-differentiable functions - use whichever is easier for a given problem.
4. Convexity upgrades a *local* first-order-necessary stationary point into a *global* minimizer, because the tangent-plane inequality then holds over the whole domain, not just a neighborhood.

## Optional exercises

1. Prove algebraically (as suggested on the slide) that the chord inequality for $f$ convex implies $f(x)\ge f(x_o)+\nabla f(x_o)^{\mathrm T}(x-x_o)$, then confirm your proof numerically for a function of your choice.
2. Find a pair of convex sets whose union *is* convex. What special relationship do they need to have?
3. Is $f_3$ (indefinite Hessian) concave anywhere, convex anywhere, or neither over all of $\mathbb R^2$? Use the chord test restricted to different rays through the origin to check.